In [ ]:
!pip install discord.py nest_asyncio pandas openai scikit-surprise
import csv
import os
import time
import json
import random
import discord
from discord.ext import commands
import nest_asyncio
import pandas as pd
import openai
from surprise import SVD, Dataset, Reader

MY_TOKEN = 'TestToken'
secret_key = 'TestSceret'

client = openai.AsyncOpenAI(api_key=secret_key)

intents = discord.Intents.default()
intents.message_content = True

bot = commands.Bot(command_prefix='!', intents=intents)

user_sessions = {}

if os.path.exists('movies.csv'):
    movies_df = pd.read_csv('movies.csv')
    movies_df['movieId'] = movies_df['movieId'].astype(str)
else:
    movies_df = pd.DataFrame(columns=['movieId', 'title', 'genres'])

movie_system_instructions = """You are acting as a middle-man type interface between the front end user typing in Discord to a Discord bot, and a backend machine learning algorithm that recommends movies.
Your job is to listen to what the users are asking and then provide a movie title they are curious about.
If someone asks anything different than whether they would like a movie, say "I can only provide assistance finding out information about movie recommendations"."""

def search_movie(partial_movie_name):
    if movies_df.empty or 'title' not in movies_df.columns:
        return "Movie database is empty or not loaded properly."

    query = partial_movie_name.strip()
    matches = movies_df[movies_df['title'].str.contains(query, case=False, na=False)]
    matches = matches.drop_duplicates(subset=['movieId'])

    if matches.empty:
        return f"No movies matching '{query}' found."

    results = []
    for _, row in matches.head(10).iterrows():
        results.append(f"ID: {row['movieId']} | Title: {row['title']}")
    return "\n".join(results)

def rate(userid, movieId, rating):
    current_timestamp = int(time.time())
    file_exists = os.path.exists('ratings.csv')
    with open('ratings.csv', 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['userId', 'movieId', 'rating', 'timestamp'])
        writer.writerow([str(userid), str(movieId), float(rating), current_timestamp])
    return f"Rated {movieId}"

def get_top_movies(userid, num_movies=5):
    userid = str(userid)
    if not os.path.exists('ratings.csv'):
        return "No ratings found"

    ratings_df = pd.read_csv('ratings.csv')
    ratings_df['userId'] = ratings_df['userId'].astype(str)
    ratings_df['movieId'] = ratings_df['movieId'].astype(str)

    user_ratings = ratings_df[ratings_df['userId'] == userid]
    watched_ids = set(user_ratings['movieId'].values)

    if len(ratings_df) < 5:
        return "Not enough data"

    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)
    algo = SVD()
    algo.fit(data.build_full_trainset())

    unwatched = movies_df[~movies_df['movieId'].isin(watched_ids)]
    predictions = []

    for m_id in unwatched['movieId'].values:
        est = algo.predict(userid, str(m_id)).est
        predictions.append((m_id, est))

    predictions.sort(key=lambda x: x[1], reverse=True)

    output = []
    for m_id, est in predictions[:num_movies]:
        title = movies_df[movies_df['movieId'] == str(m_id)]['title'].values[0]
        output.append(f"{m_id} - {title}")

    return "\n".join(output)

tools = [
    {
        "type": "function",
        "name": "get_top_movies",
        "description": "Recommend top movies for the user. The user is inferred from the Discord context.",
        "parameters": {
            "type": "object",
            "properties": {
                "num_movies": {
                    "type": "integer",
                    "description": "Number of movies to recommend",
                    "default": 5
                }
            },
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "search_movie",
        "description": "Search for movies that contain a given string in their title.",
        "parameters": {
            "type": "object",
            "properties": {
                "partial_movie_name": {
                    "type": "string",
                    "description": "A part of the movie name to search for"
                }
            },
            "required": ["partial_movie_name"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "rate",
        "description": "Add a rating entry to the ratings dataset.",
        "parameters": {
            "type": "object",
            "properties": {
                "movieId": {
                    "type": "string",
                    "description": "Movie ID to rate"
                },
                "rating": {
                    "type": "number",
                    "description": "Rating between 0.5 and 5.0"
                }
            },
            "required": ["movieId", "rating"],
            "additionalProperties": False
        }
    }
]

@bot.event
async def on_ready():
    print(f'{bot.user} is online!')

bot.remove_command('ask')
bot.remove_command('flip')
bot.remove_command('rps')

@bot.command()
async def flip(ctx):
    """Flips a coin."""
    result = random.choice(["Heads", "Tails"])
    await ctx.send(f"The coin landed on: **{result}**!")

@bot.command()
async def rps(ctx, choice: str = None):
    """Play Rock, Paper, Scissors."""
    if not choice:
        await ctx.send("Please choose rock, paper, or scissors! Example: `!rps rock`")
        return

    user_choice = choice.lower().strip()
    choices = ["rock", "paper", "scissors"]

    if user_choice not in choices:
        await ctx.send("Invalid choice!")
        return

    bot_choice = random.choice(choices)

    if user_choice == bot_choice:
        outcome = "It's a tie!"
    elif (user_choice == "rock" and bot_choice == "scissors") or \
         (user_choice == "paper" and bot_choice == "rock") or \
         (user_choice == "scissors" and bot_choice == "paper"):
        outcome = "You win!"
    else:
        outcome = "You lose!"

    await ctx.send(f"You picked **{user_choice}**, I picked **{bot_choice}**. {outcome}")

@bot.command()
async def ask(ctx, *, question: str = None):
    """Ask the AI movie bot."""
    if not question:
        await ctx.send("Please provide a question.")
        return

    message_sender = str(ctx.author.id)
    prev_id = user_sessions.get(message_sender)

    try:
        if prev_id:
            response = await client.responses.create(
                model="gpt-5-nano",
                instructions=movie_system_instructions,
                tools=tools,
                previous_response_id=prev_id,
                input=question
            )
        else:
            response = await client.responses.create(
                model="gpt-5-nano",
                instructions=movie_system_instructions,
                tools=tools,
                input=question
            )

        tool_calls = [item for item in response.output if item.type == "function_call"]

        while tool_calls:
            tool_outputs = []

            for tool_call in tool_calls:
                function_name = tool_call.name

                if isinstance(tool_call.arguments, str):
                    arguments = json.loads(tool_call.arguments)
                else:
                    arguments = tool_call.arguments

                if function_name == "get_top_movies":
                    print("getting recommended movies...")
                    output = get_top_movies(message_sender, **arguments)
                elif function_name == "search_movie":
                    print(f"searching {arguments.get('partial_movie_name')}...")
                    output = search_movie(**arguments)
                elif function_name == "rate":
                    print(f"rating {arguments.get('movieId')} {arguments.get('rating')} stars...")
                    output = rate(message_sender, **arguments)
                else:
                    output = f"Unknown function {function_name}"

                print("giving OpenAI function output:\n", output)

                tool_outputs.append({
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": str(output)
                })

            response = await client.responses.create(
                model="gpt-5-nano",
                tools=tools,
                previous_response_id=response.id,
                input=tool_outputs
            )

            tool_calls = [item for item in response.output if item.type == "function_call"]

        user_sessions[message_sender] = response.id

        final_answer = response.output_text

        if len(final_answer) > 2000:
            for i in range(0, len(final_answer), 1900):
                await ctx.send(final_answer[i:i+1900])
        else:
            await ctx.send(final_answer)

    except Exception as e:
        print("Error:", e)
        await ctx.send("Error")

nest_asyncio.apply()
bot.run(MY_TOKEN)

2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
2026-08-22 03:25:04 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2026-08-22 03:25:05 INFO     discord.gateway Shard ID None has

AIML_BOT#2982 is online!
searching Toy Story...
giving OpenAI function output:
 ID: 1 | Title: Toy Story (1995)
ID: 3114 | Title: Toy Story 2 (1999)
ID: 78499 | Title: Toy Story 3 (2010)
rating 1 5 stars...
giving OpenAI function output:
 Rated 1
getting recommended movies...
giving OpenAI function output:
 32 - Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
47 - Seven (a.k.a. Se7en) (1995)
50 - Usual Suspects, The (1995)
111 - Taxi Driver (1976)
246 - Hoop Dreams (1994)
247 - Heavenly Creatures (1994)
260 - Star Wars: Episode IV - A New Hope (1977)
293 - Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
296 - Pulp Fiction (1994)
308 - Three Colors: White (Trzy kolory: Bialy) (1994)
